In [3]:
import time
import numpy as np
import psi4

def diag_lps(diag, A, nel):
    Fp = psi4.core.triplet(A, diag, A, True, False, True)
    nbf = A.shape[0]
    Cp = psi4.core.Matrix(nbf, nbf)
    eigvecs = psi4.core.Vector(nbf)
    Fp.diagonalize(Cp, eigvecs, psi4.core.DiagonalizeOrder.Ascending)
    C = psi4.core.doublet(A, Cp, False, False)
    Cocc = psi4.core.Matrix(nbf, 1)    
    Cocc.np[:] = np.sqrt(nel) * C.np[:, :1]
    D = psi4.core.doublet(Cocc, Cocc, False, True)   
    return D

def Vpot_init(build_superfunctional, wfn, alias, vname, restricted=True):
    sup = build_superfunctional(alias, restricted)[0]
    sup.set_deriv(1)
    sup.allocate()
    Vpot = psi4.core.VBase.build(wfn.basisset(), sup, vname)
    return Vpot

def Vpot_builder(Vpot, D, V, D_half):
    D_half.copy(D)
    D_half.scale(0.5)
    Vpot.set_D([ D_half ])
    Vpot.compute_V([ V ])
    e = Vpot.quadrature_values()['FUNCTIONAL']
    return e, V

def lps_solver(maxiter, alias1, alias2, alias3, lam, mol, damp, READ=True):
    
    E_conv = 1.0e-5
    D_conv = 1.0e-5
    
    wfn = psi4.core.Wavefunction.build(mol, psi4.core.get_global_option("BASIS"))
    mints = psi4.core.MintsHelper(wfn.basisset())
    
    nbf = wfn.nso()
    nel = wfn.nalpha() + wfn.nbeta()

    print('Number of basis functions:   %d' % nbf)

    build_superfunctional = psi4.driver.dft.build_superfunctional
    D_half = psi4.core.Matrix(nbf, nbf)

    VPpot = Vpot_init(build_superfunctional, wfn, alias1, "RV", restricted=True)
    VPpot.initialize()
    VP_null = psi4.core.Matrix(nbf, nbf)

    VXCpot = Vpot_init(build_superfunctional, wfn, alias2, "RV", restricted=True)
    VXCpot.initialize()
    VXC_null = psi4.core.Matrix(nbf, nbf)

    VvWpot = Vpot_init(build_superfunctional, wfn, alias3, "RV", restricted=True)
    VvWpot.initialize()
    VvW_null = psi4.core.Matrix(nbf, nbf)

    V = mints.ao_potential()
    T = mints.ao_kinetic()
    H = T.clone()
    H.add(V)
    I = np.asarray(mints.ao_eri())
    A = mints.ao_overlap()
    A.power(-0.5, 1.e-14)
    F = psi4.core.Matrix(nbf, nbf)
    VG = psi4.core.Matrix(nbf, nbf)
    
    D = diag_lps(H, A, nel)
    if READ:
        D = D_GUESS
    
    Enuc = mol.nuclear_repulsion_energy()
    Eold = 0.0
    
    print('\nStarting SCF iterations:')
    t = time.time()
   
    print("\n    Iter            Energy            Delta E         dRMS\n")
    for SCF_ITER in range(1, maxiter + 1):
    
        D_old = D
        
        J = np.einsum('pqrs,rs->pq', I, D.np, optimize=True)
        J = psi4.core.Matrix.from_array(J)
        F.copy(H)
        F.axpy(1.0, J)

        pau_e, VP = Vpot_builder(VPpot, D, VP_null, D_half)
        xc_e, VXC = Vpot_builder(VXCpot, D, VXC_null, D_half)
        vw_e, VvW = Vpot_builder(VvWpot, D, VvW_null, D_half)

        VG.copy(VP)
        VG.axpy(1.0, VXC)
        VG.axpy((lam - 1.0), VvW)

        g_e = pau_e + xc_e + ( lam - 1.0 ) * vw_e 

        SCF_E = H.vector_dot(D)
        SCF_E += 0.5 * J.vector_dot(D)
        SCF_E += g_e
        SCF_E += Enuc

        F.axpy(1.0, VG)
        D = diag_lps(F, A, nel)
        
        D_diff = D.clone()
        D_diff.subtract(D_old)
        dRMS = D_diff.rms()

        D.scale(1.0 - damp)
        D.axpy(damp, D_old)

        print('SCF Iter%3d: % 18.8f   % 1.5E   % 1.5E'
              % (SCF_ITER, SCF_E, (SCF_E - Eold), dRMS))
        
        if (abs(SCF_E - Eold) < E_conv and dRMS < D_conv):
            break
    
        Eold = SCF_E
        
        if SCF_ITER == maxiter:
            SCF_D = D
            print("\nWARNING ! SCF did not converge. The final values are printed")
            return SCF_E, SCF_D, SCF_ITER
    
    SCF_D = D
    
    print('\nTotal time for SCF iterations: %.3f seconds ' % (time.time() - t))

    return SCF_E, SCF_D, SCF_ITER

In [7]:
psi4.core.clean_options()
psi4.core.clean()
psi4.core.set_output_file('output.dat', False)

mol = psi4.geometry("""
units bohr
0 2
H
symmetry c1
""")
psi4.set_options({'basis': 'Chan2001', 
                 'DFT_SPHERICAL_POINTS': 6,
                  'DFT_RADIAL_POINTS': 1000})

Pauli = {
    "name": "Pauli",
    "x_functionals": {"LDA_X": {"alpha": 0.00}},
    "c_functionals": {"LDA_K_TF": {"alpha": 1.00}}
}
XC = {
    "name": "XC",
    "x_functionals": {"LDA_X": {"alpha": 1.00}},
    "c_functionals": {"LDA_C_VWN": {"alpha": 0.00}}
}
vW = {
    "name": "vW",
    "x_functionals": {"LDA_X": {"alpha": 0.00}},
    "c_functionals": {"GGA_K_VW": {"alpha": 1.00}}
}

damp = 0.0

SCF_E, D, SCF_ITER = lps_solver(2000, Pauli, XC, vW, 1.0, mol, damp, READ=False)
print('\nFinal SCF energy: %.4f hartree' % SCF_E)

Number of basis functions:   19

Starting SCF iterations:

    Iter            Energy            Delta E         dRMS

SCF Iter  1:        -0.11114489   -1.11145E-01    5.34185E-02
SCF Iter  2:        -0.14317541   -3.20305E-02    5.32754E-02
SCF Iter  3:        -0.12111143    2.20640E-02    5.15482E-02
SCF Iter  4:        -0.14531334   -2.42019E-02    5.15410E-02
SCF Iter  5:        -0.12192418    2.33892E-02    5.14123E-02
SCF Iter  6:        -0.14548921   -2.35650E-02    5.14117E-02
SCF Iter  7:        -0.12199442    2.34948E-02    5.14006E-02
SCF Iter  8:        -0.14550442   -2.35100E-02    5.14006E-02
SCF Iter  9:        -0.12200052    2.35039E-02    5.13996E-02
SCF Iter 10:        -0.14550574   -2.35052E-02    5.13996E-02
SCF Iter 11:        -0.12200105    2.35047E-02    5.13995E-02
SCF Iter 12:        -0.14550586   -2.35048E-02    5.13995E-02
SCF Iter 13:        -0.12200110    2.35048E-02    5.13995E-02
SCF Iter 14:        -0.14550587   -2.35048E-02    5.13995E-02
SCF Iter 15: 

KeyboardInterrupt: 